Keras tuner-decide number of hidden layers and the neuron in neural network

In [2]:
!pip install keras-tuner
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from keras_tuner.tuners import RandomSearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 13.8 MB/s eta 0:00:00


In [4]:
df=pd.read_csv('Real_Combine.csv')

In [5]:
df.head()

,T,TM,Tm,SLP,H,VV,V,VM,PM 2.5
0,7.4,9.8,4.8,1017.6,93.0,0.5,4.3,9.4,219.720833
1,7.8,12.7,4.4,1018.5,87.0,0.6,4.4,11.1,182.187500
2,6.7,13.4,2.4,1019.4,82.0,0.6,4.8,11.1,154.037500
3,8.6,15.5,3.3,1018.7,72.0,0.8,8.1,20.6,223.208333
4,12.4,20.9,4.4,1017.3,61.0,1.3,8.7,22.2,200.645833


In [6]:
X=df.iloc[:,: -1]#independent features
y=df.iloc[:,-1]#dependent features

In [7]:
def build_model(hp):
  model= keras.Sequential()
  for i in range(hp.Int('num_layers', min_value=2, max_value=20, step=1)):
    model.add(layers.Dense(units=hp.Int('units_'+str(i),
                                        min_value=32, # Corrected keyword
                                        max_value=512, # Corrected keyword
                                        step=32),
                           activation='relu'))
  # Add the output layer after all hidden layers
  model.add(layers.Dense(1,activation='linear'))

  # Compile the model after defining its architecture
  model.compile( # Corrected typo: complie -> compile
      optimizer=keras.optimizers.Adam(
          hp.Choice('learning_rate', values=[1e-2,1e-3,1e-4])),
      loss='mean_absolute_error',
      metrics=['mean_absolute_error'])
  return model

In [8]:
tuner=RandomSearch(
    build_model,
    objective='val_mean_absolute_error',
    max_trials=5,
    executions_per_trial=3,
    directory='project',
    project_name='Air Quality Index'
)

In [9]:
tuner.search_space_summary()

Search space summary
Default search space size: 4
num_layers (Int)
{'default': None, 'conditions': [], 'min_value': 2, 'max_value': 20, 'step': 1, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
units_1 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0001], 'ordered': True}


In [10]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.3,random_state=42)

In [11]:
# Combine X_train and y_train to drop NaNs concurrently and ensure alignment
train_data_combined = pd.concat([X_train, y_train], axis=1).dropna()
X_train_clean = train_data_combined.iloc[:, :-1] # Features
y_train_clean = train_data_combined.iloc[:, -1] # Target

# Combine X_test and y_test to drop NaNs concurrently and ensure alignment
test_data_combined = pd.concat([X_test, y_test], axis=1).dropna()
X_test_clean = test_data_combined.iloc[:, :-1] # Features
y_test_clean = test_data_combined.iloc[:, -1] # Target

tuner.search(X_train_clean, y_train_clean,
             epochs=5,
             validation_data=(X_test_clean, y_test_clean))

Trial 5 Complete [00h 00m 23s]
val_mean_absolute_error: 63.73613484700521

Best val_mean_absolute_error So Far: 46.313262939453125
Total elapsed time: 00h 01m 55s


In [12]:
tuner.results_summary()

Results summary
Results in project/Air Quality Index
Showing 10 best trials
Objective(name="val_mean_absolute_error", direction="min")

Trial 1 summary
Hyperparameters:
num_layers: 5
units_0: 96
units_1: 352
learning_rate: 0.01
units_2: 32
units_3: 32
units_4: 32
Score: 46.313262939453125

Trial 2 summary
Hyperparameters:
num_layers: 17
units_0: 512
units_1: 96
learning_rate: 0.01
units_2: 224
units_3: 128
units_4: 416
units_5: 32
units_6: 32
units_7: 32
units_8: 32
units_9: 32
units_10: 32
units_11: 32
units_12: 32
units_13: 32
units_14: 32
units_15: 32
units_16: 32
Score: 49.44803365071615

Trial 0 summary
Hyperparameters:
num_layers: 2
units_0: 128
units_1: 352
learning_rate: 0.0001
Score: 62.943555196126304

Trial 3 summary
Hyperparameters:
num_layers: 9
units_0: 512
units_1: 192
learning_rate: 0.0001
units_2: 256
units_3: 320
units_4: 288
units_5: 320
units_6: 384
units_7: 416
units_8: 224
units_9: 160
units_10: 96
units_11: 512
units_12: 160
units_13: 416
units_14: 160
units_15: 